# RoboTwin × LingBot-VLA-v2 on ROCm

This notebook is the interactive entry point for the `external-data` image. Run cells selectively: model serving and evaluation are long-running GPU workloads. The platform-provided data must be available at `/models/robotwin-persistent`. Runtime outputs are written to `/workspace/runtime`.

In [ ]:
from pathlib import Path
import os, signal, subprocess, time, urllib.request

ROBOTWIN = Path('/RoboTwin')
RUNTIME = Path('/workspace/runtime')
PYTHON = '/opt/robotwin-env/bin/python'
MODEL = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/models/robbyant_lingbot-vla-v2-6b-robotwin/checkpoints/global_step_50000/hf_ckpt'
os.environ.setdefault('AITER_TRITON_ONLY', '1')
os.environ.setdefault('FLASH_ATTENTION_TRITON_AMD_ENABLE', 'TRUE')
os.environ['PYTHONPATH'] = '/opt/aiter' + (':' + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')
RUNTIME.joinpath('outputs/logs').mkdir(parents=True, exist_ok=True)
print('RoboTwin:', ROBOTWIN)
print('Model:', MODEL)

## 1. Environment and mounted-data check

In [ ]:
required = [
    ROBOTWIN / 'assets/objects/objaverse/list.json',
    ROBOTWIN / 'data/demo_clean',
    ROBOTWIN / 'data/lerobot',
    MODEL / 'model.safetensors.index.json',
]
for path in required:
    assert path.exists(), f'Missing: {path}'
subprocess.run([PYTHON, '-c', "import torch; print('torch=', torch.__version__, 'hip=', torch.version.hip, 'gpu=', torch.cuda.is_available(), 'count=', torch.cuda.device_count())"], check=True)

### Expected svulkan2 warning on AMD

During SAPIEN initialization, the following messages are expected on ROCm/AMD and can be ignored if Vulkan rendering and evaluation continue:

```text
[svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
[svulkan2] [error] Failed to initialize denoiser
```

svulkan2 is probing its optional NVIDIA CUDA denoiser; RoboTwin continues through the Vulkan renderer. Investigate `/dev/dri`, `PYOPENGL_PLATFORM=egl`, and `vulkaninfo --summary` only if the process subsequently exits or camera images are missing.

## 2. Start the model server on port 13400

Stop any existing process using port 13400 before running this cell. The server log is written under `/workspace/runtime/outputs/logs`.

In [ ]:
server_log = RUNTIME / 'outputs/logs/official_server.log'
server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(server_log), 'False', str(MODEL),
]
server_log_handle = server_log.open('ab')
SERVER_PROCESS = subprocess.Popen(server_command, cwd=ROBOTWIN, stdout=server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('server pid:', SERVER_PROCESS.pid, 'log:', server_log)
for _ in range(600):
    if SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Server exited with code {SERVER_PROCESS.returncode}; inspect {server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Server did not become ready; inspect {server_log}')

## 3. Run 10 closed-loop `adjust_bottle` episodes

> **Important: the following two lines do not indicate an evaluation failure and can be ignored.** svulkan2 is probing an optional CUDA denoiser that is available only on NVIDIA systems. These messages are expected on AMD ROCm, and RoboTwin will continue with Vulkan rendering:
>
> ```text
> [2026-08-26 12:03:21.547] [svulkan2] [error] CUDA Error: cudaErrorInsufficientDriver
> [2026-08-26 12:03:21.547] [svulkan2] [error] Failed to initialize denoiser
> ```
> Investigate the Vulkan and device configuration only if the evaluation process subsequently exits or camera images are missing.

In [ ]:
EVAL_EPISODES = 10
eval_command = [
    PYTHON, str(ROBOTWIN / 'scripts/eval_policy_xpolicylab.py'),
    '--task_name', 'adjust_bottle', '--task_config', 'demo_clean',
    '--policy_name', 'LingBot-VLA-v2', '--protocol', 'lingbot_vla_v2',
    '--host', '127.0.0.1', '--port', '13400', '--device_id', '0',
    '--seed', '0', '--test_num', str(EVAL_EPISODES), '--expert_check', 'false',
    '--eval_batch', 'false',
]
eval_env = os.environ.copy()
eval_env.update(ROBOTWIN_DISABLE_CUROBO='1', ROBOTWIN_EE_PLANNER='mplib', PYOPENGL_PLATFORM='egl')
base_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
base_eval_elapsed = time.perf_counter() - base_eval_started
print(f'base evaluation total: {base_eval_elapsed:.2f}s ({base_eval_elapsed / 60:.2f} min)')
print(f'base evaluation average per episode: {base_eval_elapsed / EVAL_EPISODES:.2f}s')

### 3.1 Optional: all 50 clean tasks, 50 episodes per task, on 1/2/4 GPUs

This is the full 2,500-episode closed-loop benchmark, not a smoke test. It is disabled by default to avoid accidentally starting a multi-hour run; set `RUN_OFFICIAL_FULL_BENCHMARK = True` to enable it. Set `BENCHMARK_GPU_COUNT` to `1`, `2`, or `4`. One model server and one RoboTwin evaluation worker are launched per GPU; a shared task queue dynamically assigns the next unfinished task to the first free GPU. The cell stops the single server from Section 2 before starting the benchmark and stops all benchmark servers when it finishes.

Time guidance from the completed 50-task run (video and task mix can change the result):

| GPUs | Time |
|---:|---:|
| 1 | about **25 h 27 min** |
| 2 | about **13 h 40 min** |
| 4 | about **6 h 50 min** |

The estimates assume one model replica per GPU, the same model/checkpoint, `demo_clean`, 50 episodes, and broadly similar server/video settings. They are not linear model-throughput benchmarks: task lengths differ substantially, so the final long task determines the tail. The run writes one log per task plus `task_times.tsv`, `events.log`, and completion markers under `/workspace/runtime/outputs/<BENCHMARK_RUN_NAME>`. Set a new run name for an independent rerun.

In [ ]:
RUN_OFFICIAL_FULL_BENCHMARK = False
BENCHMARK_GPU_COUNT = 4  # Supported values: 1, 2, or 4
BENCHMARK_EPISODES = 50
BENCHMARK_RUN_NAME = f'clean50x50_{BENCHMARK_GPU_COUNT}gpu'
BENCHMARK_RESUME = True

if RUN_OFFICIAL_FULL_BENCHMARK:
    if BENCHMARK_GPU_COUNT not in (1, 2, 4):
        raise ValueError('BENCHMARK_GPU_COUNT must be 1, 2, or 4')

    # The reusable benchmark script launches one model server per GPU.
    if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
        SERVER_PROCESS.wait(timeout=30)
        server_log_handle.close()
        print('single model server stopped before the multi-GPU benchmark')

    benchmark_script = (
        ROBOTWIN
        / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    )
    benchmark_command = [
        PYTHON, str(benchmark_script),
        '--gpu-count', str(BENCHMARK_GPU_COUNT),
        '--episodes', str(BENCHMARK_EPISODES),
        '--run-name', BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
    ]
    if BENCHMARK_RESUME:
        benchmark_command.append('--resume')

    subprocess.run(benchmark_command, cwd=ROBOTWIN, env=os.environ.copy(), check=True)
else:
    print('Skipped the long official-model 50-task benchmark')


## 4. LoRA training, checkpoint merge, and inference

Choose `GPU_COUNT` as 1, 2, or 4 and set `TRAIN_STEPS` explicitly. The default 100 steps is a smoke test for the complete pipeline, not an expectation of useful fine-tuning quality. Start with 1,000–5,000 steps for an experiment and select the final value from held-out closed-loop evaluation. The effective global batch size remains 4. This cell stops the model server started above before training so it releases GPU memory.

In [ ]:
if 'SERVER_PROCESS' in globals() and SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(SERVER_PROCESS.pid), signal.SIGTERM)
    SERVER_PROCESS.wait(timeout=30)
    server_log_handle.close()
    print('model server stopped before training')

GPU_COUNT = 1  # Supported values: 1, 2, 4
TRAIN_STEPS = 100  # Smoke test; increase to e.g. 1000 or 5000 for fine-tuning
if GPU_COUNT not in (1, 2, 4):
    raise ValueError('GPU_COUNT must be 1, 2, or 4')
if TRAIN_STEPS <= 0:
    raise ValueError('TRAIN_STEPS must be positive')
gradient_accumulation_steps = 4 // GPU_COUNT
training_root = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin'
source_root = training_root / 'source/lingbot-vla-v2'
training_yaml = training_root / 'training/reproduction_100steps/lingbotvla_cli.yaml'
training_output = RUNTIME / f'outputs/reproduction_{TRAIN_STEPS}steps'
train_command = [
    PYTHON, '-m', 'torch.distributed.run', '--standalone',
    f'--nproc-per-node={GPU_COUNT}', '-m', 'tasks.vla.train_lingbotvla',
    str(training_yaml),
    '--train.data_parallel_shard_size', str(GPU_COUNT),
    '--train.gradient_accumulation_steps', str(gradient_accumulation_steps),
    '--train.max_steps', str(TRAIN_STEPS),
    '--train.save_steps', str(TRAIN_STEPS),
    '--train.output_dir', str(training_output),
]
train_env = os.environ.copy()
train_env['HIP_VISIBLE_DEVICES'] = ','.join(str(i) for i in range(GPU_COUNT))
train_env.pop('ROCR_VISIBLE_DEVICES', None)
train_env.pop('CUDA_VISIBLE_DEVICES', None)
training_started = time.perf_counter()
subprocess.run(train_command, cwd=source_root, env=train_env, check=True)
training_elapsed = time.perf_counter() - training_started
print(f'LoRA training total: {training_elapsed:.2f}s ({training_elapsed / 60:.2f} min, {training_elapsed / 3600:.2f} h)')
print(f'LoRA wall-clock average per optimizer step: {training_elapsed / TRAIN_STEPS:.2f}s')

### 4.1 Merge the LoRA checkpoint

In [ ]:
checkpoint = training_output / f'checkpoints/global_step_{TRAIN_STEPS}'
MERGED = training_output / f'merged_checkpoint/global_step_{TRAIN_STEPS}/hf_ckpt'
assert checkpoint.exists(), f'Missing checkpoint: {checkpoint}'
merge_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/merge_lora_dcp.py'),
    '--checkpoint', str(checkpoint),
    '--training-output', str(training_output),
    '--base-model', str(MODEL), '--output', str(MERGED),
    '--rank', '8', '--alpha', '16',
]
subprocess.run(merge_command, cwd=ROBOTWIN, check=True)
assert (MERGED / 'model.safetensors.index.json').exists(), f'Merge output is incomplete: {MERGED}'
print('merged model:', MERGED)

### 4.2 Start the merged LoRA model on port 13400

The original server was stopped before training, so the merged model reuses the same endpoint.

In [ ]:
merged_log = RUNTIME / 'outputs/logs/merged_server.log'
merged_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(merged_log), 'False', str(MERGED),
]
merged_log_handle = merged_log.open('ab')
MERGED_SERVER_PROCESS = subprocess.Popen(merged_server_command, cwd=ROBOTWIN, stdout=merged_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
print('merged server pid:', MERGED_SERVER_PROCESS.pid, 'log:', merged_log)
for _ in range(600):
    if MERGED_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Merged server exited with code {MERGED_SERVER_PROCESS.returncode}; inspect {merged_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('merged server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Merged server did not become ready; inspect {merged_log}')

### 4.3 Validate the merged LoRA model with 10 closed-loop episodes

In [ ]:
merged_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
merged_eval_elapsed = time.perf_counter() - merged_eval_started
print(f'merged evaluation total: {merged_eval_elapsed:.2f}s ({merged_eval_elapsed / 60:.2f} min)')
print(f'merged evaluation average per episode: {merged_eval_elapsed / EVAL_EPISODES:.2f}s')

### 4.4 Optional: run all 50 tasks against the merged LoRA model

This is a long 2,500-episode evaluation: allow about **6 h 50 min on four GPUs**, **13 h 40 min on two GPUs**, or **25 h 27 min on one GPU**. The shared script starts one model server per GPU, so it first stops the single merged-model server above. `--resume` skips tasks that already have a completion marker in the same run directory. Change `LORA_BENCHMARK_RUN_NAME` when you want a completely independent rerun instead of resuming existing results.


In [ ]:
RUN_LORA_FULL_BENCHMARK = False
LORA_BENCHMARK_GPU_COUNT = 4
LORA_BENCHMARK_RUN_NAME = f'lora_{TRAIN_STEPS}steps_clean50x50_{LORA_BENCHMARK_GPU_COUNT}gpu'

if RUN_LORA_FULL_BENCHMARK:
    if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
        MERGED_SERVER_PROCESS.wait(timeout=30)
        merged_server_log_handle.close()
        print('single merged-model server stopped before the full benchmark')
    benchmark_script = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    subprocess.run([
        PYTHON, str(benchmark_script),
        '--gpu-count', str(LORA_BENCHMARK_GPU_COUNT),
        '--episodes', '50',
        '--model-path', str(MERGED),
        '--run-name', LORA_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
        '--resume',
    ], cwd=ROBOTWIN, env=os.environ.copy(), check=True)
else:
    print('Skipped the long merged-LoRA 50-task benchmark')


### 4.5 Optional: stop the merged LoRA model server

In [ ]:
if 'MERGED_SERVER_PROCESS' in globals() and MERGED_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(MERGED_SERVER_PROCESS.pid), signal.SIGTERM)
    MERGED_SERVER_PROCESS.wait(timeout=30)
    merged_log_handle.close()
    print('merged model server stopped')
else:
    print('no merged server started by this notebook kernel')

## 5. Full-parameter SFT, checkpoint conversion, and inference

This is independent from the LoRA workflow above: it sets `use_lora=false` and updates all trainable model parameters. `FULL_SFT_GPU_COUNT` supports 1, 2, or 4 and defaults to 4. The script keeps global batch 4 by using gradient accumulation 4, 2, or 1 respectively. Start with `FULL_SFT_STEPS = 1`; each full DCP checkpoint directory is roughly 70 GB in total across all ranks, not 70 GB per GPU.

Under FSDP2, `FULL_SFT_ENABLE_FULL_SHARD=True` means `reshard_after_forward=True`: each wrapped module is sharded again after forward, lowering peak VRAM at the cost of another all-gather before backward. `False` keeps that module's full parameters through backward, which may be faster but generally uses more peak VRAM. It does not disable FSDP or replicate the entire model on every GPU. Keep `True` unless a one-step comparison proves that `False` fits safely.

The available four-GPU configurations are: (1) the default full-shard run below with shard/global batch 4; (2) the same run with no reshard for the communication-versus-memory comparison; and (3) a full-shard run with `enable_fp32` and `use_future_image` enabled. They are alternatives, not consecutive training stages.

In [ ]:
# Start with 1 step. GPU count may be 1, 2, or 4; the default is 4.
FULL_SFT_GPU_COUNT = 4
FULL_SFT_STEPS = 1
FULL_SFT_SAVE_STEPS = FULL_SFT_STEPS
FULL_SFT_ENABLE_FULL_SHARD = True  # Recommended lower-memory default
FULL_SFT_OUTPUT = RUNTIME / f'outputs/full_sft_{FULL_SFT_GPU_COUNT}gpu_{FULL_SFT_STEPS}steps'

for process_name in ('SERVER_PROCESS', 'MERGED_SERVER_PROCESS'):
    process = globals().get(process_name)
    if process is not None and process.poll() is None:
        os.killpg(os.getpgid(process.pid), signal.SIGTERM)
        process.wait(timeout=30)

gpu_count = int(subprocess.check_output(
    [PYTHON, '-c', 'import torch; print(torch.cuda.device_count())'], text=True
).strip())
assert FULL_SFT_GPU_COUNT in (1, 2, 4), 'FULL_SFT_GPU_COUNT must be 1, 2, or 4'
assert gpu_count >= FULL_SFT_GPU_COUNT, f'Full SFT requested {FULL_SFT_GPU_COUNT} GPUs, found {gpu_count}'

full_sft_env = os.environ.copy()
full_sft_env.update({
    'GPU_COUNT': str(FULL_SFT_GPU_COUNT),
    'MAX_STEPS': str(FULL_SFT_STEPS),
    'SAVE_STEPS': str(FULL_SFT_SAVE_STEPS),
    'ENABLE_FULL_SHARD': str(FULL_SFT_ENABLE_FULL_SHARD).lower(),
    'OUTPUT_DIR': str(FULL_SFT_OUTPUT),
})
full_sft_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/training/train_full_sft.sh')
]
full_sft_started = time.perf_counter()
subprocess.run(full_sft_command, cwd=ROBOTWIN, env=full_sft_env, check=True)
full_sft_elapsed = time.perf_counter() - full_sft_started
print(f'full SFT total: {full_sft_elapsed:.2f}s ({full_sft_elapsed / 60:.2f} min)')
print('checkpoint:', FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}')

### 5.1 Convert the full-SFT DCP checkpoint

Full SFT has no LoRA adapter to merge. This step gathers the distributed DCP model state and writes a directly loadable Hugging Face checkpoint. Do not use `merge_lora_dcp.py` here.

In [ ]:
FULL_SFT_CHECKPOINT = FULL_SFT_OUTPUT / f'checkpoints/global_step_{FULL_SFT_STEPS}'
FULL_SFT_MODEL = FULL_SFT_OUTPUT / f'merged_checkpoint/global_step_{FULL_SFT_STEPS}/hf_ckpt'
assert FULL_SFT_CHECKPOINT.exists(), f'Missing checkpoint: {FULL_SFT_CHECKPOINT}'
full_sft_convert_command = [
    PYTHON, str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/convert_full_sft_dcp.py'),
    '--checkpoint', str(FULL_SFT_CHECKPOINT),
    '--training-output', str(FULL_SFT_OUTPUT),
    '--output', str(FULL_SFT_MODEL),
]
convert_started = time.perf_counter()
subprocess.run(full_sft_convert_command, cwd=ROBOTWIN, check=True)
convert_elapsed = time.perf_counter() - convert_started
assert (FULL_SFT_MODEL / 'model.safetensors.index.json').exists(), f'Conversion is incomplete: {FULL_SFT_MODEL}'
print(f'full-SFT conversion total: {convert_elapsed:.2f}s ({convert_elapsed / 60:.2f} min)')
print('full-SFT model:', FULL_SFT_MODEL)

### 5.2 Start and validate the converted full-SFT model on port 13400

In [ ]:
full_sft_server_log = RUNTIME / 'outputs/logs/full_sft_server.log'
full_sft_server_command = [
    'bash', str(ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/launch_official_server.sh'),
    '0', '13400', str(full_sft_server_log), 'False', str(FULL_SFT_MODEL),
]
full_sft_server_log_handle = full_sft_server_log.open('ab')
FULL_SFT_SERVER_PROCESS = subprocess.Popen(full_sft_server_command, cwd=ROBOTWIN, stdout=full_sft_server_log_handle, stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(600):
    if FULL_SFT_SERVER_PROCESS.poll() is not None:
        raise RuntimeError(f'Full-SFT server exited with code {FULL_SFT_SERVER_PROCESS.returncode}; inspect {full_sft_server_log}')
    try:
        urllib.request.urlopen('http://127.0.0.1:13400/healthz', timeout=2).read()
        print('full-SFT server ready: 127.0.0.1:13400')
        break
    except Exception:
        time.sleep(2)
else:
    raise TimeoutError(f'Full-SFT server did not become ready; inspect {full_sft_server_log}')

full_sft_eval_started = time.perf_counter()
subprocess.run(eval_command, cwd=ROBOTWIN, env=eval_env, check=True)
full_sft_eval_elapsed = time.perf_counter() - full_sft_eval_started
print(f'full-SFT evaluation total: {full_sft_eval_elapsed:.2f}s ({full_sft_eval_elapsed / 60:.2f} min)')
print(f'full-SFT evaluation average per episode: {full_sft_eval_elapsed / EVAL_EPISODES:.2f}s')

### 5.3 Optional: run all 50 tasks against the converted full-SFT model

This is also a long 2,500-episode evaluation: allow about **6 h 50 min on four GPUs**, **13 h 40 min on two GPUs**, or **25 h 27 min on one GPU**. The shared script first replaces the single full-SFT server with one model server per selected GPU. `--resume` continues the same run by skipping completed task markers. Change `FULL_SFT_BENCHMARK_RUN_NAME` for an independent rerun of the same checkpoint.


In [ ]:
RUN_FULL_SFT_FULL_BENCHMARK = False
FULL_SFT_BENCHMARK_GPU_COUNT = 4
FULL_SFT_BENCHMARK_RUN_NAME = f'full_sft_{FULL_SFT_STEPS}steps_clean50x50_{FULL_SFT_BENCHMARK_GPU_COUNT}gpu'

if RUN_FULL_SFT_FULL_BENCHMARK:
    if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
        os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
        FULL_SFT_SERVER_PROCESS.wait(timeout=30)
        full_sft_server_log_handle.close()
        print('single full-SFT server stopped before the full benchmark')
    benchmark_script = ROBOTWIN / 'experiments/lingbot_vla_v2_6b_robotwin/scripts/run_clean_benchmark.py'
    subprocess.run([
        PYTHON, str(benchmark_script),
        '--gpu-count', str(FULL_SFT_BENCHMARK_GPU_COUNT),
        '--episodes', '50',
        '--model-path', str(FULL_SFT_MODEL),
        '--run-name', FULL_SFT_BENCHMARK_RUN_NAME,
        '--runtime-dir', str(RUNTIME),
        '--robotwin-root', str(ROBOTWIN),
        '--resume',
    ], cwd=ROBOTWIN, env=os.environ.copy(), check=True)
else:
    print('Skipped the long full-SFT 50-task benchmark')


### 5.4 Optional: stop the full-SFT model server

In [ ]:
if 'FULL_SFT_SERVER_PROCESS' in globals() and FULL_SFT_SERVER_PROCESS.poll() is None:
    os.killpg(os.getpgid(FULL_SFT_SERVER_PROCESS.pid), signal.SIGTERM)
    FULL_SFT_SERVER_PROCESS.wait(timeout=30)
    full_sft_server_log_handle.close()
    print('full-SFT model server stopped')
else:
    print('no full-SFT server started by this notebook kernel')